In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/us-birth/annotated/checkpoints/post_cell_9.pickle

trying: ['birth_data']


me:  6
trying: ['time']
me:  0
trying: ['years2check']
me:  9
trying: ['year']
me:  9
trying: ['orig_output']
me:  12
trying: ['yearly_data']
me:  9
trying: ['pd']
me:  0
trying: ['factor']
me:  2
trying: ['sns']
me:  0
trying: ['plt']
me:  0
trying: ['np']
me:  0
Declaring variable time
Declaring variable pd
Declaring variable sns
Declaring variable plt
Declaring variable np
Declaring variable factor
Declaring variable birth_data
Declaring variable years2check
Declaring variable year
Declaring variable yearly_data
Declaring variable orig_output


In [4]:
%%RecordEvent
%%time
### cell 10 ###

# Compute quartiles on the GPU
gquart = birth_data['births'].quantile([0.25, 0.5, 0.75])
mu = gquart.loc[0.5]
sig = 0.74 * (gquart.loc[0.75] - gquart.loc[0.25])

# Compute bounds and filter via boolean indexing (all on GPU)
lower_bound = mu - 5 * sig
upper_bound = mu + 5 * sig
births = birth_data[(birth_data['births'] > lower_bound) & (birth_data['births'] < upper_bound)]

# Final results
birth_data.shape, births.sample(5, random_state=0)

CPU times: user 193 ms, sys: 52.3 ms, total: 245 ms
Wall time: 245 ms


((7773500, 6),
        year  month  day gender  births  decade
 5069   1975      8   16      M    4342    1970
 1750   1971      4   13      M    5366    1970
 2793   1972      8   25      F    4797    1970
 13703  1987      2   24      F    5396    1980
 7636   1978     12   31      M    4202    1970)

In [5]:
%Checkpoint /home/dias-benchmarks/new_notebooks/us-birth/rewritten/q20_rewrite/checkpoints/post_cell_10_try_4.pickle

migration speed (bps): 790016130.8064617
---------------------------
variables to migrate:
yearly_data 48
lower_bound 32
sns 72
years2check 120
plt 72
factor 28
year 28
mu 32
np 72
time 72
birth_data 48
sig 32
gquart 48
births 48
upper_bound 32
pd 72
orig_output 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/new_notebooks/us-birth/rewritten/q20_rewrite/checkpoints/post_cell_10_try_4.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['birth_data']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['birth_data']
Intermediate variables ['factor']
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['birth_data']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables ['birth_data']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['birth_data']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables ['birth_data']
Intermediate variables []
Future variables []
Modified dataframes
  birth_data
    Input columns: set()
    Changed columns: set()
    Created columns: {'decade'}
    Deleted columns: set()
======= Cell 6 =======
Input

In [7]:

with open("/home/dias-benchmarks/new_notebooks/us-birth/opt_cell_exec_info_10_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[10], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
births_opt = births
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/us-birth/annotated/checkpoints/post_cell_10.pickle
assert compare_df(births_opt, births)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
